# GMRESについて

## 概要

GMRES(Generalized Minimal Redidual Method)は、連立一次方程式 $Ax=b$ の解 $x$ を反復計算によって近似的に求める手法である。

GMRESでは Krylov部分空間を作成し、その部分空間の中から残差 $r = b - Ax$ のノルムが最小となる解を探索する。これにより、大規模な連立一次方程式を解く際に、元の問題よりも低次元の空間で解を探索することが出来る。

GMRESを理解するための主な要素として、Krylov部分空間、Arnoldi法、最小二乗問題、QR分解、Givens回転がある。本ノートではそれらを順に説明し、最後にPythonによるGMRESの実装を行う。

### Krylov部分空間

解く問題は $A x = b$ という連立一次方程式である。<br>
ここで A は $n \times n$ 次元の行列、$x, b$ はそれぞれ $n$ 次元のベクトルとする。

GMRESでは近似初期解を $x_0$ として 近似解を $x_m = x_0 + z_m$ としたときにこの$z_m$ をKrylov部分空間 $\mathcal{K}_m = \text{span}\{r_0, A r_0, A^2 r_0, \cdots A^{(m-1)} r_0\}$ の中で繰り返し計算で探すことで、元の連立一次方程式を効率よく解く。


このKrylov部分空間だけを見てみても連立方程式を解くことイメージが繋がりにくいので、この先の Arnoldi 法でつながりを見ていく。

###  Arnoldi法

Arnoldi法は正規直交ベクトルと、上ヘッセンベルグ行列をStepを進めるごとに更新していく。

#### Step 0

近似初期解を$x_0$ として 初期残差を $r_0 = b - Ax_0$ とする。<br>
$v_1 = r_0 / ||r_0||$ として、$r_0$ 方向の単位ベクトル $v_1$ を作る

このとき $K_1= \text{span}\{v_1 \} = \text{span}\{r_0\}$<br>
よって、$v_1$ がKrylov部分空間の基底になる。

#### Step 1

$h_{11} = v_1^T A v_1$ として、$A v_1$ と $v_1$ の内積から、$A v_1$が基底 $v_1$にどれだけ乗っているかを計算する。ここで$A v_1$ はベクトル $v_1$ に線形変換 $A$ を作用させて得られる新しいベクトルである。

$w = A v_1 - h_{11} v_1$ として、$A v_1$方向から$v_1$要素を引き、$v_1$に直交するベクトルを作成する。

$h_{21} = ||w||$ として、$v_1$に直交するベクトルの大きさを求める。

$v_2 = w / h_{21}$ として、$v_1$に直交する単位ベクトル $v_2$ を求める。

このとき 新たな基底として$v_2$を採用し $K_2= \text{span}\{v_1 , v_2\}$となる。<br>
よって、$v_1, v_2$がKrylov部分空間の基底になる。

Arnoldi法では、各ステップで $A v_j$ を既存の基底と新しい基底の線形結合で表す。このとき表れる係数 $h_{ij}$ を列ごとに並べた行列が上ヘッセンベルグ行列 $H$ である。例えばStep 1 では次の関係式となる。

$$
\boxed{
A v_1 = h_{11} v_1 + h_{21} v_2
}
$$

この係数を一列目に並べると、

$$
\bar{H}_1 = \begin{bmatrix}
h_{11} \\ h_{21}
\end{bmatrix}
$$

-------------------------------------------------------------

##### Krylov部分空間 $\{r_0 , A r_0 \}$ との関係

部分空間 $\text{span}$ の定義は
$\text{span}\{x_1, x_2\}= \{c_1 x_1 + c_2 x_2 | c_1 , c_2 \in \mathcal{R} \}$であり、$\text{span}\{x_1, x_2\}$ は任意の $c1,c2$ を係数としたベクトル $c_1 x_1 + c_2 x_2$ の線形結合の集合である。

集合$A,B$ の二つがあり、$A=B$を示すには、 $A \subseteq B$ と $B \subseteq A$ を示す必要がある。

ここでは、$\text{span}\{v_1, v_2\}$ = $\text{span}\{r_0, A r_0 \}$を示す。

##### $\text{span}\{v_1, v_2\}$ $\subseteq$ $\text{span}\{r_0, A r_0 \}$ を示す

$\alpha = 1/||r_0||$とすると、$v_1 = \alpha r_0$である。

また、$v_2 = (A v_1 - h_{11} v_1)/h_{21}$ であり、$\beta = - \alpha h_{11}/h_{21}, \gamma = \alpha / h_{21}$ とすると、$v_2 = \beta r_0 + \gamma A r_0$になる。<br>
任意のベクトルは 任意の $c_1, c_2$に対して $c_1 v_1 + c_2 v_2$ と書ける。<br>
$v_1,v_2$を代入すると以下となる。

$$
c_1 \alpha r_0 + c_2 (\beta r_0 + \gamma A r_0) = (c_1 \alpha + c_2 \beta) r_0 + c_2 \gamma A r_0
$$

よって $c_1 v_1 + c_2 v_2 \in \text{span}\{ r_0, A r_0\}$ であり、

$$
\text{span}\{v_1, v_2\} \subseteq \text{span}\{r_0, A r_0 \}
$$

##### $\text{span}\{r_0, A r_0\}$ $\subseteq$ $\text{span}\{v_1, v_2 \}$ を示す

$v_2 = (A v_1 - h_{11} v_1)/h_{21}$より、$A v_1 = h_{11} v_1 + h_{21} v_2$ となる。<br>
$v_1 = \alpha r_0$ より、$A v_1 = \alpha A r_0$ である。
よって $A r_0 = (h_{11} v_1 + h_{21} v_2) /\alpha$ となり $A r_0$ は $v_1, v_2$の線形結合で表すことが出来る。
任意のベクトルは任意の $d_1, d_2$ に対して $d_1 r_0 + d_2 A r_0$ と書ける。<br>
$r_0, A r_0$ を代入すると以下となる。

$$
d_1 r_0 + d_2 A r_0 = \frac{d_1}{\alpha} v_1 + \frac{d_2 (h_{11} v_1 + h_{21} v_2)}{\alpha} =  \left(\frac{d_1}{\alpha} +\frac{d_2 h_{11}}{\alpha}\right) v_1 + \frac{d_2 h_{21}}{\alpha} v_2 
$$

よって、$d_1 r_0 + d_2 A r_0 \in \text{span} \{v_1, v_2\}$であり、

$$
\text{span}\{r_0, A r_0\} \subseteq \text{span}\{v_1, v_2 \}
$$

以上より、以下を示せる。

$$
\text{span}\{v_1, v_2\} = \text{span}\{r_0, A r_0 \}
$$

-------------------------------------------------------------

### Step 2

$h_{12} = v_1^T A v_2$ として、$A v_2$と $v_1$ の内積から、$A v_2$が基底$v_1$にどれだけ乗っているかを計算する。<br>
$h_{22} = v_2^T A v_2$ として、$A v_2$と $v_2$ の内積から、$A v_2$が基底$v_2$にどれだけ乗っているかを計算する。<br>
ここで$A v_2$ ベクトル $v_2$ に線形変換$A$を作用させて得られる新たなベクトルである。

$w = A v_2 - h_{12} v_1 - h_{22} v_2$ として、 $A v_2$から$v_1, v_2$要素を引き、$v_1, v_2$に直交するベクトルを作成する。

$h_{32} = ||w||$ として、$v_1, v_2$に直交するベクトルの大きさを求める。<br>
$v_3 = w / h_{32}$ として、$v_1, v_2$に直交する単位ベクトル $v_3$ を求める。

このとき 新たな基底として $v_3$を採用し、$K_3= \text{span}\{v_1 , v_2, v_3\}$<br>
よって、$v_1, v_2, v_3$がKrylov部分空間の基底になる。

このStepで得られる $A v_2$の展開は以下となる。

$$
\boxed{
A v_2 = h_{12} v_1 + h_{22} v_2 + h_{32} v_3
}
$$

この式の係数を上ヘッセンベルグ行列の第２列に追加すると

$$
\bar{H}_2 = \begin{bmatrix}
h_{11} & h_{12} \\
h_{21} & h_{22} \\
0 & h_{32}
\end{bmatrix}
$$

ここでも Step 1 と同様に $\text{span} \{ v_1, v_2, v_3\} = \text{span} \{r_0, A r_0, A^2 r_0 \}$ と示せるが割愛する。

ここでこれまで得られた式を再掲すると以下となる。

$$
\begin{aligned}
A v_1 &= h_{11} v_1 + h_{21} v_2 \\
A v_2 &= h_{12} v_1 + h_{22} v_2 + h_{32} v_3
\end{aligned}
$$

これを横に並べると、左辺は

$$
\begin{bmatrix} A v_1 & A v_2 \end{bmatrix} = A \begin{bmatrix} v_1 & v_2 \end{bmatrix} 
$$

右辺は以下となる。

$$
\begin{bmatrix}
h_{11} v_1 + h_{21} v_2 & h_{12} v_1 + h_{22} v_2 + h_{32} v_3
\end{bmatrix} = 
\begin{bmatrix}v_1 & v_2 & v_3 \end{bmatrix}
\begin{bmatrix}h_{11} & h_{12} \\ h_{21} & h_{22} \\ 0 & h_{32}  \end{bmatrix}
$$

ここで$V_2 = \begin{bmatrix} v_1 & v_2 \end{bmatrix},V_3 = \begin{bmatrix} v_1 & v_2 & v_3 \end{bmatrix}$ また$\bar{H}_2$を以下とすると、

$$
\bar{H}_2 = 
\begin{bmatrix}h_{11} & h_{12} \\ h_{21} & h_{22} \\ 0 & h_{32}  \end{bmatrix}
$$

次のような $A$ と $\bar{H}$ に関する関係式を得る。

$$
\boxed{
A V_2 = V_3 \bar{H}_2
}
$$

#### Arnoldi法の一般化

Step0 で初期近似解と $x_0$ と初期残差 $r_0$ を設定し、Arnoldi法を$m$ ステップまで実行する。<br>
第$j$ステップ ($j = 1,\cdots, m$)では、次の計算を行う。

$$
\begin{aligned}
h_{ij} &= v_i^T A v_j \space , \space (i = 1,\cdots , j)\\
w &= A v_j - \sum_{i=1}^j h_{ij} v_i \\
h_{j+1,j} &= ||w|| \\
v_{j+1} &= \frac{w}{h_{j+1, j}}
\end{aligned}
$$

これを$j=1,\cdots , m$ について繰り返す。

そしてKrylov部分空間は $\text{span}\{v_1, \cdots , v_{m+1}\} = \text{span}\{r_0 , \cdots , A^m r_0\}$ となる。

また、上ヘッセンベルグ行列は以下となる。

$$
\bar{H}_m = 
\begin{bmatrix}
h_{11} & h_{12} & h_{13} & \cdots & h_{1m} \\
h_{21} & h_{22} & h_{23} & \cdots & h_{2m} \\
0 & h_{32} & h_{33} & \cdots & h_{3m} \\
0 & 0 &  h_{43} & \cdots & h_{4m} \\
\vdots & \vdots & \ddots & \ddots & \vdots \\
0 & 0 & \cdots & h_{m,m-1} & h_{mm} \\
0 & 0 & \cdots & 0 & h_{m+1, m}
\end{bmatrix}
$$

$\bar{H}_m$の第$j$列には、$A v_j = \sum_{i=1}^{j+1} h_{ij}v_i$ の係数 $h_{ij}$が格納される。$A v_j$ の展開には$v_{j+1}$より後の基底は現れないため、第$j$列の $j+2$ 行目以降はゼロとなる。この構造が上ヘッセンベルグ形である。

各ステップで得られた $A v_j = \sum_{i=1}^{j+1} h_{ij} v_i$ を横に並べて整理すると、Step 2 で見たように以下の関係を得る。

$$
\boxed{
A V_m = V_{m+1} \bar{H}_m
}
$$

$\bar{H}_m$ は $(m+1) \times m$ の行列であることに注意する。


Arnoldi法で生成される基底 $v_1, v_2 , \cdots$ はすべて $\mathbf{R}^n$ のベクトルであり、互いに正規直交する。$n$次元空間内で線形独立なベクトルは最大でも$n$本であるため、Krylov部分空間の次元およびArnoldi法の反復回数の上限は $n$ 回である。

ただし途中で

$$
h_{j+1, j} = 0
$$ 

となった場合、それ以上の独立な基底を生成できなくなったことを意味し、計算は打ち切られる。

実際の大規模問題では計算量を抑えるため、通常は $m \ll n$ として打ち切る。

Arnoldi法は、残差 $r_0$ から始めて、行列 $A$ を作用させることでKrylov部分空間を少しずつ拡張し、その空間の正規直交基底 $V_m$ を生成する方法である。同時に、$A$ をこの基底上で表現した小さな行列 $\bar{H}_m$ が得られ、

$$
AV_m = V_{m+1}\bar{H}_m
$$

という関係が成立する。GMRESでは、この関係を利用して、元の $n$ 次元の問題を $m$ 次元程度の小さな最小二乗問題へ変換する。

### 近似解を求めるための最小二乗問題

Arnoldi法により、初期近似解 $x_0$ から Krylov部分空間での正規直交基底 $V_m, V_{m+1}$ と 上ヘッセンベルグ行列 $\bar{H}_{m}$ が得られた。この状態から $A x = b$ の近似解 $x_m$を求める。

GMRESは繰り返し計算より解を近似的に求める。よって、Arnoldi法 $m$ 回目の近似解 $x_m$ は以下のように更新される。

$$
x_m = x_0 + z_m
$$

この近似解 $x_m$ が得られた場合の残差 $r$ は次のようになる。

$$
r = b - A x_m = b - A(x_0 +z_m) = b - A x_0 - A z_m = r_0 - A z_m
$$

$z_m$ は Krylov部分空間で表現される$x_0$から解候補 $x$ までの更新量であり、$z_m \in \text{span}\{v_1, \cdots , v_m\}$ である。これは、$z_m = y_1 v_1 + \cdots y_m v_m$ のように $z_m$ が表されることを示している。<br>
$Y=[y_1, \cdots, y_m]^T$ と置けば、$z_m = V_m Y$ と表現できる。この$Y$ が求めるべき値である。

Arnoldi法で得られた $AV_m = V_{m+1} \bar{H}_m$ より

$$
r = r_0 - A V_m Y = r_0 - V_{m+1} \bar{H}_m Y  
$$

ここで $\beta = ||r_0||$ とすると、 $v_1 = r_0 / \beta$ より $r_0 = \beta v_1$となる。

また、$e_1 = [1, 0, \cdots, 0]^T \in \mathcal{R}^{m+1}$ とすると、$v_1$ は次のようになる。

$$
v_1 = V_{m+1} e_1 = [v_1, \cdots, v_{m+1}]\begin{bmatrix} 1 \\ 0 \\ \vdots \\ 0 \end{bmatrix}
$$

よって 残差 $r$ は次のようになる。

$$
r = \beta V_{m+1} e_1 - V_{m+1} \bar{H}_m Y
$$

$\beta$ はスカラーであるので、次のようにまとめられる。

$$
r =  V_{m+1} \left(\beta e_1 - \bar{H}_m Y \right)
$$

この残差の大きさ $||r||^2 = r^T r$ を近似解 $x_m$ により最小化したいという問題になる。

$V_{m+1}$ は正規直交基底であるため、$V_{m+1}^T V_{m+1} = \mathbf{I}$ となる。よって以下となる。

$$
\begin{aligned}
||r||^2 =& \left(V_{m+1} \left(\beta e_1 - \bar{H}_m Y \right)\right)^T\left(V_{m+1} \left(\beta e_1 - \bar{H}_m Y \right)\right)\\
=& \left(\beta e_1 - \bar{H}_m Y \right)^T V_{m+1}^T V_{m+1} \left(\beta e_1 - \bar{H}_m Y \right) \\
=&  \left(\beta e_1 - \bar{H}_m Y \right)^T \left(\beta e_1 - \bar{H}_m Y \right) \\
=& ||\beta e_1 - \bar{H}_m Y ||^2
\end{aligned}
$$

よって2ノルム($||x|| = \sqrt{\sum_i x_i^2}$) は

$$
||r|| = ||\beta e_1 - \bar{H}_m Y ||
$$

求めたいものは $Y = [y_1, \cdots, y_m]^T$ であり、この$Y$ による最小化を考える。

$$
\boxed{
\min\limits_Y ||\beta e_1 - \bar{H}_m Y||
}
$$

ここで $Y$ が求まれば、$z_m = V_m y$ と $x_m = x_0 + z_m$ より近似解 $x_m$ を求めることが出来る。

この最適化問題は未知数が $Y \in \mathcal{R}^m$ のみであり、元のn次元の連立一次方程式は、$m$次元の小さな最小化問題へ変換されたことになる。


### $Y$ の計算

解きべきは以下の最小化問題である。

$$
\min\limits_Y ||\beta e_1 - \bar{H}_m Y||
$$

もし残差 $||r||$ を完全にゼロにできる $Y$ が存在すれば、

$$
\bar{H}_m Y = \beta e_1
$$

のようになるが、$\bar{H}_m$ は $(m+1) \times m$ の縦長行列であるため、一般にはすべての式を同時に満たす $Y$ が存在するとは限らない。<br>

そのため、残差ノルムを最小化にする最小二乗問題として解く。

この最小化問題を効率的に解くために、$\bar{H}_m$ をQR分解して上三角行列へ変換する。上三角行列へ変換することで、$Y$ は後退代入によって求めることが出来る。

### QR分解

行列 $A$ を正規直交行列 $Q$ と 上三角行列 $R$ に分解することを QR分解という。

$$
A = QR
$$

正規直交行列は $Q^T Q = \mathbf{I}$ となる性質がある。よって、

$$
Q^T A = R
$$

とすると、上三角行列 $R$ を求めることが出来る。

これを残差2ノルムの式に当てはめて考える。

$$
||r|| = ||\beta e_1 - \bar{H}_m Y ||
$$

今 $\bar{H}_m = QR$ と分解したときに、$Q^T \bar{H}_m = R$ となる関係がある。<br>
$Q^T Q= \mathbf{I}$ であるため、以下のように $Q$ をあるベクトル $x$ にかけても以下のように2ノルムを変化させない。

$$
||Qx||^2 = (Qx)^T(Qx) = x^T Q^T Q x = x^T x = ||x||^2 \rightarrow ||Qx|| = ||x||
$$ 

よって、

$$
||Q^T \ r|| = ||Q^T\left(\beta e_1 - \bar{H}_m Y\right) || = ||Q^T \beta e_1 - R Y ||
$$

これより、以下の最小化問題となり、

$$
\min\limits_Y ||Q^T \beta e_1 - R Y ||
$$

ここで、$Q^T \beta e_1 = g$として、次のように解くことが出来る。

$$
\boxed{
R Y = g
}
$$

ここで、$R \in \mathcal{R}^{(3,2)}$の上三角行列、$Y =[y_1, y_2]^T$, $g = [g_1, g_2, g_3]^T$ とすると、以下の式となり、

$$
\begin{bmatrix}
r_{11} & r_{12} \\
0 & r_{22} \\
0 & 0 
\end{bmatrix}
\begin{bmatrix}
y_1 \\ y_2
\end{bmatrix}
=
\begin{bmatrix}
g_1 \\ g_2 \\ g_3
\end{bmatrix}
$$

上2行から後退代入で解($y_1, y_2$)が求める。

$$
\begin{aligned}
r_{11} y_1 + r_{12} y_2 &= g_1 \\
r_{22} y_2 &= g_2
\end{aligned} \rightarrow
\begin{aligned}
y_2 &= g_2 / r_{22} \\
y_1 &= \left(g_1 - g_2 r_{12}/r_{22} \right) / r_{11}  \\
\end{aligned}
$$

後退代入により上2行は厳密に満たされるため、最後の1行だけが残差になる。

$$
\min\limits_Y ||g - R Y || = \min\limits_Y || [0 , 0 , g_3]^T || = \sqrt{g_3 \times g_3} = |g_3|
$$

従って、残差2ノルム $||r||$ は$|g_3|$ を評価すれば良い。

このようにQR分解によって$\bar{H}$は上三角行列に変換されるため、求めたい $Y$ を後退代入によって容易に求めることが出来る。しかし、ステップ毎にQR分解を毎回最初から計算すると計算量が大きい。そこでGMRESでは、Givens回転を用いて、各ステップでQR分解を逐次更新する。


### Givens回転

各ステップでGivens回転を行う前に、Givens回転行列とは何かを説明する

Givens回転は以下のようなもので、$k$ は第k列の対角線直下の要素をゼロにするための回転を表す。空白の箇所はゼロである。$c$は$\cos$、$s$は$\sin$ を示す。

$$
G_k = 
\begin{bmatrix}
I_{k-1} &  &\\ 
& c & s& \\
& -s & c& \\
& & &  I 
\end{bmatrix}
$$

この$G_k$が正規直交行列であることを、$G_k^T G_k = \mathbf{I}$ で示す。簡単のため、$G_k$のサイズを4、$k=2$とする。

$$
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & c & -s & 0 \\
0 & s & c & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & c & s & 0 \\
0 & -s & c & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}=
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & c^2 + s^2 & cs - cs & 0 \\
0 & -cs + cs & c^2 + s^2 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix} =
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$


このGivens回転を使い、例えば、$H=[h_1, h_2]^T$ という行列をから$h_2$を消去することを考える。

ここで、$l = \sqrt{h_1^2 + h_2^2}$ とすると、座標($h_1, h_2$)から、$h_1$は直角三角形の底辺(X方向)、$h_2$は高さ(Y方向)とみなせる。このとき、

$$
c = \cos(\theta) = \frac{h_1}{l}, \quad s = \sin(\theta) = \frac{h_2}{l}
$$

となるような回転角 $\theta$ を選び、次の$G$ を $H$ に掛ける。

$$
G = \begin{bmatrix}
c & s \\
-s & c
\end{bmatrix}
$$

すると以下のように$H$を$G$で回転することになる。$G$は時計回りに$H$を回転させるため、直角三角形の高さ(Y方向)が消え、X方向が残る。

$$
G H= \begin{bmatrix}
c & s \\
-s & c
\end{bmatrix}
\begin{bmatrix}
h_1 \\ h_2
\end{bmatrix} =
\begin{bmatrix}
c h_1 + s h_2 \\
-s h_1 + c h_2
\end{bmatrix} =
\begin{bmatrix}
 \dfrac{h_1^2 + h_2^2}{l} \\
- \dfrac{h_1 h_2}{l}  + \dfrac{h_1 h_2}{l}
\end{bmatrix} 
=\begin{bmatrix}
l \\ 0
\end{bmatrix}
$$

これがGivens回転の基本的な考え方である。


$\bar{H}_2$ をGivens回転して、上三角行列 $R$ を作ることを考える。

$$
\bar{H}_2 = \begin{bmatrix}
h_{11} & h_{12} \\
h_{21} & h_{22} \\
0 & h_{32}
\end{bmatrix}
$$

Step1 で 1列目の$h_{21}$ を消去し、Step 2 で 2列目の$h_{32}$ を消去する。

Step1 

さっきの直角三角形の考え方で、$h_{11}$ を底辺(X方向)、$h_{21}$ を高さ(Y方向)とする。

すると、$l_1 = \sqrt{h_{11}^2 + h_{21}^2}$として、次の $G_1$ を$\bar{H}_2$ の左側からかける。

$$
G_1 = 
\begin{bmatrix}
h_{11} /l_1 & h_{21} / l_1 & 0\\
- h_{21} /l_1 & h_{11} / l_1 & 0 \\
0 & 0 & 1
\end{bmatrix}
$$

すると、次のように1列目の $h_{21}$ が消える。また2列目も同時に回転するため、変化する要素を $h \rightarrow g$ としている。

$$
G_1 \bar{H}_2 = 
\begin{bmatrix}
h_{11} /l_1 & h_{21} / l_1 & 0\\
- h_{21} /l_1 & h_{11} / l_1 & 0\\
0 & 0 & 1
\end{bmatrix}
\begin{bmatrix}
h_{11} & h_{12} \\
h_{21} & h_{22} \\
0 & h_{32}
\end{bmatrix} =
\begin{bmatrix}
l_1 & g_{12} \\
0 & g_{22} \\
0 & h_{32}
\end{bmatrix} 
$$

Step 2

次に2列目の $h_{32}$ を消去することを考える。

($g_{22}, h_{32}$) を直角三角形の底辺と高さと考え、$l_2 = \sqrt{g_{22}^2 + h_{32}^2}$ を長さとする。

次の $G_2$ により、$G_1 \bar{H}_2$ を更に回転させる。

$$
G_2 = 
\begin{bmatrix}
1 & 0 & 0 \\
0 & g_{22} /l_2 & h_{32} / l_2 \\
0 & - h_{32} /l_2 & g_{22} / l_2 \\
\end{bmatrix}
$$

すると、次のように$h_{32}$ が消えて、上三角行列が現れる。

$$
G_2 G_1 \bar{H}_2 = 
\begin{bmatrix}
1 & 0 & 0 \\
0 & g_{22} /l_2 & h_{32} / l_2 \\
0 & - h_{32} /l_2 & g_{22} / l_2 \\
\end{bmatrix}
\begin{bmatrix}
l_1 & g_{12} \\
0 & g_{22} \\
0 & h_{32}
\end{bmatrix} 
=
\begin{bmatrix}
l_1 & g_{12} \\
0 & l_2 \\
0 & 0
\end{bmatrix}
$$

ここで、これまで計算したGivens回転の行列$G$は正規直交行列であるため、QR分解のQに相当する。<br>
よって、上記で行っていたことは、$H=G^T R$のQR分解の関係から$GH = R$を計算していたことになる。

$$
G_2 G_1 H = R \rightarrow Q^T H = R
$$

各Stepで計算される上ヘッセンベルグ行列の1列目から順番にQR分解を行うと計算量が大きくなる。そこで、GMRESでは、前のステップで得られたGivens回転を再利用し、新しく追加された列に対して1回だけGivens回転を追加することでQR分解を逐次更新する。

### GMRESの流れ

$A x = b$ の方程式を解く問題を考える。ここで$A \in \mathcal{R}^{n \times n}$である。

Step 0を行ったうえで、Step 3 までを説明を行いながら式で記述する。
そして、Step m ではアルゴリズムとしてまとめるために、より一般的な記述で説明を行う。

#### Step 0
初期近似解 $x_0$ として、$r_0 = b - A x_0$ を計算する。

$$
\boxed{
\beta = ||r_0|| 
}
$$

$$
\boxed{
v_1 = r_0 / \beta
}
$$

#### Step 1

Arnoldi法 1回目の処理で次を実行する。

$$
\begin{aligned}
h_{11} &= v_1 ^ T A v_1 \\
w &= A v_1 - h_{11} v_1 \\
h_{21} &= ||w|| \\
v_2 &= w / h_{21}
\end{aligned}
$$

次に上ヘッセンベルグ行列を得る。

$$
\bar{H}_1 =
\begin{bmatrix}
h_{11} \\ h_{21}
\end{bmatrix}
$$

解く問題は以下の式である。

$$
\boxed{
RY = g
}
$$

Givens回転前の右辺を次のようにする。

$$
g^{(0)} = \beta e_1 = \beta \begin{bmatrix} 1 \\ 0 \end{bmatrix}
$$

Givens回転の係数を次のようにし、

$$
\begin{matrix}
l_1 = \sqrt{h_{11}^2 + h_{21}^2} \ ,& c_1 = h_{11} / l_1 \ ,& s_1 = h_{21}/ l_1
\end{matrix}
$$

Givens回転行列を次のように構成し、

$$
G_1 = \begin{bmatrix}
c_1 & s_1 \\
-s_1 & c_1
\end{bmatrix}
$$

$\bar{H}_1$ へかけると上三角行列 $\bar{R}_1$ を得る

$$
\bar{R}_1 = G_1 \bar{H}_1 = 
\begin{bmatrix}
l_1 \\ 0
\end{bmatrix}
$$

右辺 $g^{(0)}$ も$G_1$をかけ回転させる。右肩の"()"内の数字は、どのGivens回転行列によって変換されているかを表している。

$$
g^{(1)} =G_1 g^{(0)} = \beta \begin{bmatrix}
c_1 \\ -s_1
\end{bmatrix}
$$

ここまでの計算で、下記の式が完成している。

$$
g^{(1)} - \bar{R}_1 Y_1 = \beta \begin{bmatrix}
c_1 \\ -s_1
\end{bmatrix} - \begin{bmatrix}
l_1 \\ 0
\end{bmatrix}
y_1
$$

$Y_1$を上式の上1式はがゼロにるように選ぶことで、最後の1成分が残差になる。この残差 $||r_1|| = |\beta s_1|$ を確認して、十分に小さければここでArnoldi法を終了させ、次の最小化問題より$y_1$を求める。
残差が大きい場合次のStepへ進む。

$$
\min\limits_{y_1} || g^{(1)} - \bar{R}_1 y_1||
$$

残差部分を除外し、式を整理すると$y_1$が求まる。

$$
\beta c_1 - l_1 y_1 = 0 \rightarrow y_1 = \beta c_1 /l_1
$$

今 m=1 であるため、$V_m = [v_1]$ である。 
よって、近似解$x$は以下となる。

$$
\boxed{
x = x_0 + V_1 Y_1 = x_0 + v_1 y_1
}
$$



#### Step 2

Arnoldi法 2回目の処理で次を実行する。

$$
\begin{aligned}
h_{12} &= v_1 ^ T A v_2 \\
h_{22} &= v_2 ^ T A v_2 \\
w &= A v_2 - h_{12} v_1 - h_{22} v_2 \\
h_{32} &= ||w|| \\
v_3 &= w / h_{32}
\end{aligned}
$$


次にStep1で計算した上ヘッセンベルグ行列にGivens回転を施した行列の1つ横の列に$(h_{12},h_{22},h_{32})$を追加する。
以降、過去のGivens回転を適用済みの作業行列も便宜上 $\bar{H}$ でと表す。

$$
\bar{H}_2 =
\begin{bmatrix}
l_{1} & h_{12} \\
0 & h_{22} \\
0 & h_{32}
\end{bmatrix}
$$

この$\bar{H}_2$ にひとつ前のStepで計算したGivens回転$G_1$を左からかけることを考える。<br>
$\bar{H}_2$の1列目には$G_1$をかけた結果が入っている。そのため。2列目に$G_1$を作用させる。<br>
また、$G_1$は1,2行目を変換するものであったため、$h_{12}, h_{22}$を次のような変換をかける。

$$
\bar{H}_2[0,1] = c_1 h_{12} + s_1 h_{22}  = {h^{(1)}}_{12}\\
\bar{H}_2[1,1] = - s_1 h_{12} + c_1 h_{22} = {h^{(1)}}_{22}\\
$$

これより、Step 2 での上ヘッセンベルグ行列は次のようになる。

$$
\bar{H}_2 =
\begin{bmatrix}
l_{1} & {h^{(1)}}_{12} \\
0 & {h^{(1)}}_{22} \\
0 & h_{32}
\end{bmatrix}
$$

ここからGivens回転行列の要素は、2,3行目の要素を用いて下記となる。

$$
l_2 = \sqrt{{{h^{(1)}}_{22}}^2  + {h_{32}}^2} , \space c_2 = {{h^{(1)}}_{22}} / l_2 , \space s_2 = {h_{32}}/ l_2\\
$$

そして、Givens回転行列 $G_2$は以下となる。

$$
G_2 = \begin{bmatrix}
1 & 0 & 0 \\
0 & c_2 & s_2 \\
0 & -s_2 & c_2 \\
\end{bmatrix}
$$

$G_2$は2,3行目に対してのみ作用する、また、${h^{(1)}}_{22}$ を底辺、${h_{32}}$を高さとする直角三角形を右へ回転させ、高さをゼロにするような作用になるため、$\bar{H}_2$と$G_2$により $\bar{R}_2$ は。

$$
\bar{R}_2 = G_2 \bar{H}_2 =
\begin{bmatrix}
l_{1} & {h^{(1)}}_{12} \\
0 & l_{2} \\
0 & 0
\end{bmatrix}
$$

また、右辺は Step 1 で $g^{(1)} = G_1 \beta e$ であり、 Step 2 では $v_3$ までを考えるため $e \in \mathbf{R}^{3 \times 1}$ とする。<br>
右肩の"()"内の数字はどのGivens回転の作用によって変換させられたかを示している。$g^{(2)}$から分かる通り、数字が"(2)"や"(3)"の場合、それより下の数字のGivens回転行列がすでに作用していることを意味している。

$$
g^{(2)} = G_2 \ g^{(1)} = \beta G_2 \begin{bmatrix}  c_1 \\ -s_1 \\ 0\end{bmatrix} 
$$

以上の計算だが、ここでも $G_2$ が作用するのは2,3行目であることを考慮すると、

$$
g^{(2)} = \beta \left[
\begin{array}{r} c_1 \\ -c_2 s_1 \\ s_2 s_1
\end{array}
\right]
$$

以上により、下記の関係式が完成する。

$$
g^{(2)} - \bar{R}_2 Y_2 = 
\beta \begin{bmatrix} c_1 \\ -c_2 s_1 \\ s_2 s_1 \end{bmatrix} -
\begin{bmatrix}
l_{1} & {h^{(1)}}_{12} \\
0 & l_{2} \\
0 & 0
\end{bmatrix}
\begin{bmatrix}
y_1 \\ y_2
\end{bmatrix}
$$

$Y_2$ を上式の1行目、2行目をゼロになるように選ぶことで、最後の1成分が残差として残る。この残差 $||r_2|| = |\beta s_2 s_1|$ が十分小さければここでArnoldi法を終了させ、次の最小化問題より$Y_2$を求める。
残差が大きい場合次のStepへ進む。

$$
\min\limits_{Y_2} || g^{(2)} - \bar{R}_2 Y_2||
$$

残差部分を除外し、式を記述すると以下のように$Y_2$が求まる。

$$
\begin{aligned}
l_2 y_2 & = -\beta c_2 s_1 \\
l_1 y_1 + h^{(1)}_{12} y_2 &= \beta c_1
\end{aligned} \rightarrow
\begin{aligned}
y_2 & = -\beta c_2 s_1 / l_2 \\
y_1 &= \left(\beta c_1 + h^{(1)}_{12} y_2 \right) / l_1
\end{aligned}
$$

今 m=2 であるため、$V_m = [v_1, v_2]$ である。よって近似解$x$は以下のように求まる。

$$
\boxed{
x = x_0 + V_m Y_2 = x_0 + v_1 y_1 + v_2 y_2 
}
$$




#### Step 3

Arnoldi法 3回目の処理で次を実行する。

$$
\begin{aligned}
h_{13} &= v_1 ^ T A v_3 \\
h_{23} &= v_2 ^ T A v_3 \\
h_{33} &= v_3 ^ T A v_3 \\ 
w &= A v_3 - h_{13} v_1 - h_{23} v_2 - h_{33} v_3 \\
h_{43} &= ||w|| \\
v_4 &= w / h_{43}
\end{aligned}
$$

次にStep2で計算した上ヘッセンベルグ行列にGivens回転を施した行列の1つ横の列に$(h_{13},h_{23},h_{33}, h_{43})$を追加する。

$$
\bar{H}_3 = 
\begin{bmatrix}
l_1 & {h^{(1)}}_{12} & h_{13} \\
0 & l_2 & h_{23} \\
0 & 0 & h_{33} \\
0 & 0 & h_{43}
\end{bmatrix}
$$

この $\bar{H}_3$ の3列目にこれまでのGivens回転を施していく。

まず$G_1$により、1,2行目が次のようになる。

$$
\begin{aligned}
\bar{H}_3[0,2] &= c_1 h_{13} + s_1 h_{23} = {h^{(1)}}_{13} \\
\bar{H}_3[1,2] &= -s_1 h_{13} + c_1 h_{23} = {h^{(1)}}_{23} \\
\end{aligned}
$$

次に $G_2$により、2,3行目が次のようになる。

$$
\begin{aligned}
\bar{H}_3[1,2] &= c_2 {h^{(1)}}_{23} + s_2 h_{33} = {h^{(2)}}_{23} \\
\bar{H}_3[2,2] &= -s_2 {h^{(1)}}_{23} + c_2 h_{33} = {h^{(2)}}_{33} \\
\end{aligned}
$$

これより、これまでのGivens回転より、上ヘッセンベルグ行列は以下となる。

$$
\bar{H}_3 = 
\begin{bmatrix}
l_1 & {h^{(1)}}_{12} & {h^{(1)}}_{13} \\
0 & l_2 & {h^{(2)}}_{23} \\
0 & 0 & {h^{(2)}}_{33} \\
0 & 0 & h_{43}
\end{bmatrix}
$$

この$\bar{H}_3$の ${h^{(2)}}_{33}$ と $h_{43}$ をGivens回転 $G_3$ により回転させる。

$G_3$の要素を ${h^{(2)}}_{33}$ を底辺、$h_{43}$ を高さとして、次のようになる。

$$
l_3 = \sqrt{{{h^{(2)}}_{33}}^2+ {h_{43}}^2} , \space c_3 = {h^{(2)}}_{33} / l_3 , \space s_3 = h_{43} / l_3
$$

実際の$G_3$は 4 x 4 の行列だが、作用するのは、3,4行目のみである。
$G_3$により$\bar{H}_3$は以下の上三角行列になる。

$$
\bar{R}_3 = G_3 \bar{H}_3 = 
\begin{bmatrix}
l_1 & {h^{(1)}}_{12} & {h^{(1)}}_{13} \\
0 & l_2 & {h^{(2)}}_{23} \\
0 & 0 & l_3 \\
0 & 0 & 0
\end{bmatrix}
$$

また、右辺の$g^{(2)}$にも$G_3$を以下のように作用させ、$g^{(3)}$を求める。

$$
g^{(3)} = G_3 \ g^{(2)} = \beta \left[ \begin{array}{r}
c_1 \\ -c_2 s_1 \\ c_3 s_2 s_1 \\ -s_3 s_2 s_1
\end{array} \right]
$$

以上より下記の関係式が完成する。

$$
g^{(3)} - \bar{R}_3 Y_3 = \beta \left[ \begin{array}{r}
c_1 \\ -c_2 s_1 \\ c_3 s_2 s_1 \\ -s_3 s_2 s_1
\end{array} \right] - 
\begin{bmatrix}
l_1 & {h^{(1)}}_{12} & {h^{(1)}}_{13} \\
0 & l_2 & {h^{(2)}}_{23} \\
0 & 0 & l_3 \\
0 & 0 & 0
\end{bmatrix}
\begin{bmatrix}
y_1 \\ y_2 \\ y_3
\end{bmatrix}
$$

上式の上3行をゼロにするように選ぶことで、最後の1成分だけが残差として残る。この残差 $||r_3|| = |\beta s_3 s_2 s_1|$ が十分に小さければここでArnoldi法を終了させ、以下の最小化問題より$Y_3$を求める。そうでない場合は、次のStepへ進む。

$$
\min\limits_{Y_3} || g^{(3)} - \bar{R}_3 Y_3||
$$

残差部分を除外し、式を記述すると以下のように$Y_3$が求まる。

$$
\begin{aligned}
\beta \ c_1 &= l_1 \ y_1 + {h^{(1)}}_{12} \ y_2 + {h^{(1)}}_{13} \ y_3 \\
-\beta \ c_2 \ s_1 &= l_2 \ y_2 + {h^{(2)}}_{23} \ y_3 \\
\beta \ c_3 \ s_2 \ s_1 &= l_3 \ y_3
\end{aligned} \rightarrow
\begin{aligned}
y_3 &= \left( \beta \ c_3 \ s_2 \ s_1 \right) / l_3\\
y_2 & = -\left( \beta \ c_2 \ s_1 + {h^{(2)}}_{23} \ y_3 \right) / l_2 \\
y_1 & = \left( \beta \ c_1 -{h^{(1)}}_{12} \ y_2 - {h^{(1)}}_{13} \ y_3 \right) / l_1
\end{aligned}
$$

今 m=3 であるため、$V_m = [v_1, v_2, v_3]$ である。よって近似解$x$は以下のように求まる。

$$
\boxed{
x = x_0 + V_m Y_3 = x_0 + v_1 y_1 + v_2 y_2 + v_3 y_3 
}
$$


このStep 3 で少し考えてみると、例えば Aのサイズが100 x 100だとしても、ここで残差が十分に小さい場合、3 x 3 の上三角行列で近似解を求めることが出来るため、問題を小さい空間で解いていることが分かる。




#### Step m

Arnoldi法 m 回目の計算は次のようになる。

$$
\begin{aligned}
h_{i,m} &= {v_{i}}^T A v_{m} \space (i=(1, \ 2, \ \cdots \ m)) \\
w &= A v_{m} -\sum_{i=1}^m h_{i,m}v_i \\
h_{m+1, m} &= ||w|| \\
v_{m+1} &= w / h_{m+1, m}
\end{aligned}
$$

このStepでの上ヘッセンベルグ行列の列の要素は $h_{i,m}, \ (i=1,2, \cdots, m, m+1)$ であり、$i=m$の要素までは$i=1$から以下のように二つの要素ずつをこれまでのGivens回転$G_k, \ (k=1,2, \cdots , m-1)$ の$c_k, s_k$ を作用させていく。

``` text
h[k] = [h_{i,m}] の列
for k = 1,2, ..., m-1
    a = h[k]
    b = h[k+1]
    h[k] = c[k] a + s[k] b
    h[k+1] = -s[k] a + c[k] b
```

この処理によりこのStepでの上ヘッセンベルグ行列の要素は以下のようになる。

$$
\left[ h^{(1)}_{1,m}, \ h^{(2)}_{2,m}, \ \cdots , {h^{(m-1)}}_{m-1,m}, \ {h^{(m-1)}}_{m,m}, \ h_{m+1,m} \right]^T
$$

次にこのStepでのGivens回転の要素を計算する。

$$
l_m = \sqrt{{{h^{(m-1)}}_{m,m}}^2 + {h_{m+1,m}}^2} , \space c_m = {h^{(m-1)}}_{m,m} / l_m, \space s_m = h_{m+1,m}/l_m
$$

このGivens回転により、上ヘッセンベルグ行列の要素は以下となる。

$$
\left[ h^{(1)}_{1,m}, \ h^{(2)}_{2,m}, \ \cdots , {h^{(m-1)}}_{m-1,m}, \ l_m, \ 0 \right]^T
$$

また、このStepでの上三角行列 $\bar{R}_m$ は前回のStepの$\bar{R}_{m-1}$の右側に上記の結果を追加することで以下となる。

$$
\bar{R}_m = \begin{bmatrix}
l_1 & {h^{(1)}}_{12} & {h^{(1)}}_{13} & \cdots & {h^{(1)}}_{1,m-1} & {h^{(1)}}_{1,m}\\
0 & l_2 & {h^{(2)}}_{23} & \cdots & {h^{(2)}}_{2, m-1} &  {h^{(2)}}_{2,m} \\
0 & 0 & l_3 & \cdots & {h^{(3)}}_{3, m-1} &  {h^{(3)}}_{3,m} \\
\vdots & \vdots & \ddots & \ddots & \vdots & \vdots \\
0 & 0 & 0 & \cdots & l_{m-1} & {h^{(m-1)}}_{m-1,m} \\
0 & 0 & 0 & \cdots & 0 & l_m \\
0 & 0 & 0 & \cdots & 0 & 0
\end{bmatrix}
$$

右辺の$g^{(m)}$ は前回の$g^{(m-1)}$の末尾に0を追加し、今回の$G_m$を末尾の2要素に作用させるばよい。

$$
g^{(m)} = \beta \left[ \begin{array}{r}
c_1 \\
-c_2 \ s_1 \\
c_3 \ s_2 \ s_1 \\
-c_4 \ s_3 \ s_2 \ s_1 \\
\vdots \\
(-1)^{m+1} \ c_m \ \displaystyle\prod_{k=1}^{m-1} s_k\\
(-1)^{m} \ \displaystyle\prod_{k=1}^{m }s_k
\end{array}
\right]
$$

ここから残差 $||r_m||$ は$g^{(m)}$ の末尾の絶対値であるため、以下のようになる。

$$
||r_m|| = \left|{g^{(m)}}_{m+1} \right| = \beta \left| \prod_{k=1}^m s_k \right|
$$

この残差 $||r_m||$ が十分に小さい場合、次のように$\bar{R}_m$ と $g^{(m)}$ の最終行を除いた以下の関係より、$Y_m$を求める。そうでない場合、次のStep m+1 へ進める。

$$
\boxed{
g^{(m)}_{1:m} = R_m Y_m
}
$$

$$
\beta \left[ \begin{array}{r}
c_1 \\
-c_2 \ s_1 \\
c_3 \ s_2 \ s_1 \\
-c_4 \ s_3 \ s_2 \ s_1 \\
\vdots \\
(-1)^{m+1} \ c_m \ \displaystyle\prod_{k=1}^{m-1} s_k\\
\end{array}
\right] = \begin{bmatrix}
l_1 & {h^{(1)}}_{12} & {h^{(1)}}_{13} & \cdots & {h^{(1)}}_{1,m-1} & {h^{(1)}}_{1,m}\\
0 & l_2 & {h^{(2)}}_{23} & \cdots & {h^{(2)}}_{2, m-1} &  {h^{(2)}}_{2,m} \\
0 & 0 & l_3 & \cdots & {h^{(3)}}_{3, m-1} &  {h^{(3)}}_{3,m} \\
\vdots & \vdots & \ddots & \ddots & \vdots & \vdots \\
0 & 0 & 0 & \cdots & l_{m-1} & {h^{(m-1)}}_{m-1,m} \\
0 & 0 & 0 & \cdots & 0 & l_m \\
\end{bmatrix}
\begin{bmatrix}
y_1 \\ y_2 \\ y_3 \\ \vdots \\ y_{m-1} \\ y_m
\end{bmatrix}
$$

$y_m$から順に後退代入を行い、$Y_m = [y_1, \cdots , y_m]^T$ を求める。

$$
\begin{aligned}
y_m &= \beta (-1)^{m+1} c_m \prod_{k=1}^{m-1} s_k / l_m \\
\vdots \\
y_n &= \left( \beta (-1)^{n+1} c_n \prod_{k=1}^{n-1}s_k - \sum_{j=n+1}^m {h^{(n)}}_{n, j} \ y_j \right) / l_n \\
\vdots \\
y_1 &= \left(\beta c_1 - \sum_{j=2}^{m} {h^{(1)}}_{1,j} \ y_j\right)/l_1
\end{aligned}
$$

そして、近似解 $x$ は以下のように求まる。

$$
\boxed{
x = x_0 + V_m Y_m = x_0 + \sum_{i=1}^{m} v_i \ y_i
}
$$


### 実装

GMRESのアルゴリズムをPythonで実装する。説明では理解のため行列の形を記述しているが、実装では行列の計算は行わず、毎回のStepで前回の値を更新するようにしている。

実装では、数式上のアルゴリズムに加えて、数値計算上の安定性や異常ケースを考慮し、以下の点に注意する。

- 連立一次方程式 $Ax=b$ に対して、$A,b,x_0$ の次元が整合していることを確認する。
- 初期近似解 $x_0$ の残差 $r_0=b-Ax_0$ が十分に小さい場合は、$x_0$ を解として終了する。
- Arnoldi法の直交化には Modified Gram-Schmidt 法を用いる。各基底方向の成分を逐次除去することで、上記で説明したClassical Gram-Schmidt法より丸め誤差による直交性の崩れを抑える。
- Arnoldi法で $h_{m+1,m}=\|w\|$ が十分に小さく、新しい基底を生成できない場合は breakdown とする。ただし、すでに残差が十分小さい場合は収束として扱う。
- Givens回転で用いる
  $$
  l_m=\sqrt{{h_{m,m}}^2+{h_{m+1,m}}^2}
  $$
  が十分に小さい場合、上三角行列 $R$ の対角要素もほぼゼロとなり、通常の後退代入を安定に行えないため計算を終了する。
- 収束判定には、現在の残差だけでなく、
  $$
  \frac{\|r_m\|}{\|r_0\|}
  $$
  として初期残差からどの程度残差が減少したかを表す相対残差を用いる。

In [1]:
import numpy as np

class GMRES:

    def __init__(self, A:np.ndarray, b:np.ndarray, x0:np.ndarray):

        self.A = A
        self.b = b
        self.x0 = x0

        n, m = A.shape

        if n != m:
            raise ValueError("error: A is not square")
        if b.shape != (n,):
            raise ValueError("error: b is not compatible with A")
        if x0.shape != (n,):
            raise ValueError("error: x0 is not compatible with A")

        # 基底のサイズのゼロ行列を作成
        self.V = np.zeros((n, n+1))

        # Givens回転行列の要素を格納する配列
        self.c = np.zeros(n)
        self.s = np.zeros(n)

        # 上三角行列の要素を格納する配列
        self.R = np.zeros((n, n))

        # gベクトルの要素を格納する配列
        self.g = np.zeros(n+1)

        # Step 0
        r0 = self.b - self.A @ self.x0
        self.beta = np.linalg.norm(r0)
        v1 = r0 / self.beta
        self.V[:, 0] = v1

        self.g[0] = self.beta


    def solve(self, max_iter:int=100, rtol:float=1e-6, atol:float=1e-8) -> np.ndarray:

        if self.beta < atol: # 絶対誤差判定
            # 初期残差が十分小さい場合は、初期値を解として返す
            self.solved = True
            return self.x0
        
        max_iter = min(max_iter, self.A.shape[0])  # 最大反復回数は行列のサイズに制限される
        
        # Arnoldi process
        for k in range(max_iter):
            # Step m 

            # 逐次直交化(Modified Gram-Schmidt)を行う
            # 各基底方向の成分を1つずつ除去する Modified Gram-Schmidtのほうが、
            # 一括で直交化を行う Classical Gram-Schmidt より丸め誤差による直交性の崩れが小さい
            w = self.A @ self.V[:, k] # wの初期値 Av_k
            h = np.zeros(k+2) # 上ヘッセンベルグ行列の要素を格納する配列

            for j in range(k+1):
                h[j] = self.V[:, j].T @ w # 上ヘッセンベルグ行列の要素を計算
                w = w - h[j] * self.V[:, j] # wからv_jの成分を引いて直交化

            h_next = np.linalg.norm(w)
            h[k+1] = h_next

            breakdown = h_next < 1e-14

            # breakdownが起きた場合は、基底を新たに作ることが出来なかったため、Vへ次の基底を作らない
            if not breakdown:
                self.V[:, k+1] = w / h_next

            # これまでのGivens回転を今回作った上ヘッセンベルグ列に適用
            for i in range(k): # (k=0の場合(Arnoldi法1回目)はこのループは実行されない)
                temp = self.c[i] * h[i] + self.s[i] * h[i+1]
                h[i+1] = -self.s[i] * h[i] + self.c[i] * h[i+1]
                h[i] = temp

            # 今回のGivens回転
            l = np.hypot(h[k], h[k+1]) # sqrt(h[k]**2 + h[k+1]**2)

            if l < 1e-14:
                # 作成したKrylov部分空間・上三角行列の対角要素がほぼゼロで後退代入を安定的に実行できない
                raise ValueError("GMRES breakdown : R is singular or nearly singular")

            self.c[k] = h[k] / l
            self.s[k] = h[k+1] / l
            # 上ヘッセンベルグ行列の回転
            h[k] = l
            h[k+1] = 0

            # 上三角行列の要素を格納
            for i in range(k+1):
                self.R[i,k] = h[i]

            # gベクトルの回転
            temp = self.c[k] * self.g[k] + self.s[k] * self.g[k+1] 
            self.g[k+1] = -self.s[k] * self.g[k] + self.c[k] * self.g[k+1]
            self.g[k] = temp

            # 残差計算
            residual = abs(self.g[k+1]) # 残差のノルムを計算(絶対値)
            residual_rel = residual / self.beta # 初期残差との比で残差を計算し、残差が小さくなっていることを確認する
            # 収束判定
            if residual_rel < rtol or residual < atol: # 相対残差と絶対残差による収束判定
                # 上三角行列を後退代入して解を求める
                y = np.zeros(k+1)
                for i in range(k, -1, -1):
                    y[i] = (self.g[i] 
                            - sum(self.R[i,j] * y[j] for j in range(i+1, k+1)) # i=k の場合、 sum() の中身は空になるので0となる
                            ) / self.R[i,i]

                # 解を更新
                x = self.x0 + self.V[:, :k+1] @ y
                return x

            if breakdown:
                # breakdown:基底を新たに生成できず、かつ残差も収束していない場合
                raise ValueError("Arnoldi breakdown occurred, cannot find a new basis vector")

        # 収束しなかった場合
        raise ValueError("GMRES did not converge within the maximum number of iterations")



実装の確認

In [3]:
A = np.array([
    [ 8.0, -1.2,  0.5,  0.0,  2.0],
    [ 0.8,  6.5, -2.0,  0.3,  0.0],
    [ 0.0,  1.5,  9.0, -1.0,  0.7],
    [ 1.2,  0.0,  0.6,  4.5, -1.8],
    [-0.4,  0.9,  0.0,  2.2,  7.0]
])

b = np.array([3.7, -2.4, 5.1, 1.8, -0.9])

x0 = np.array([0.2, -0.1, 0.3, 0.0, 0.1])

gmres_solver = GMRES(A, b, x0)
x = gmres_solver.solve()

print("GMRES solution:", x)
print("numpy solution:", np.linalg.solve(A, b))
print("Residual norm:", np.linalg.norm(b - A @ x))

GMRES solution: [ 0.41879167 -0.23335008  0.63233491  0.15470673 -0.12326044]
numpy solution: [ 0.41879167 -0.23335008  0.63233491  0.15470673 -0.12326044]
Residual norm: 9.222205069512407e-16


## まとめ

GMRESは逐次的にKrylov部分空間を広げ、その部分空間中の残差を評価する。残差ノルムが十分に小さくなった時点で、小さな最小二乗問題を解くことで近似解を求める。

このように、元の大規模な連立一次方程式を直接解くのではなく、より低次元のKrylov部分空間上の小さな最小二乗問題として扱うことで、効率よく近似解を求めることが出来る。

また、実装では巨大な行列を毎回明示的に操作するのではなく、Arnoldi法で生成した基底ベクトルと、上ヘッセンベルグ行列の必要な列、Givens回転の係数 $c,s$、右辺ベクトル $g$ などの必要な要素のみを逐次更新することで、計算量を抑えることが出来る。

## 最適制御問題へのつながり

PMPでは以下の状態方程式、随伴方程式、停留条件、終端条件を満たすことで、運動方程式を考慮した最適な状態軌道と制御入力を連続時間の問題として求める。

$$
\dot{x} = H_\lambda , \ \dot{\lambda} = - H _x , \ H_u = 0, \ \lambda(T) = \Phi_x
$$

C/GMRESでは、離散時間で予測ホライゾン上の制御入力などを変数 $U$ にまとめ、PMP条件から得られる非線形方程式

$$
F(U,x,t) = 0
$$

を考える。実時間で時刻が進むごとに、この条件を満たす $U$ を追従させる。

この $U$ の更新量を求める過程では、$F(U,x,t)=0$ から得られる大規模な連立一次方程式を繰り返し解く必要がある。

C/GMRESでは、この連立一次方程式をGMRESによって効率的に解くことで、最適制御入力を実時間で更新することが出来る。